In [ ]:
import pandas as pd

# Read the first CSV file into a DataFrame
df_distinct_values = pd.read_csv("../../datasets/distinct_values_correcoes.csv")

df_distinct_values

In [ ]:
len(df_distinct_values[(df_distinct_values['Constraint Deleted'] == True)] )

In [ ]:
len(df_distinct_values[(df_distinct_values['Constraint Deprecated'] == True)] )

In [ ]:
len(df_distinct_values[(df_distinct_values['Included as Exception'] == True)] )

In [ ]:
len(df_distinct_values[(df_distinct_values['A-box wdt statement Deleted'] == True)] )

In [ ]:
len(df_distinct_values[(df_distinct_values['Indirect Correction by Exception'] == True)] )

In [ ]:
len(df_distinct_values[(df_distinct_values['Indirect Correction by Statement Deletion'] == True)] )

In [ ]:
df_distinct_values[
    (df_distinct_values['Constraint Deleted'] == False)
    & (df_distinct_values['Constraint Deprecated'] == False)
    & (df_distinct_values['Included as Exception'] == False)
    & (df_distinct_values['A-box wdt statement Deleted'] == False) 
    & (df_distinct_values['Indirect Correction by Exception'] == False) 
    & (df_distinct_values['Indirect Correction by Statement Deletion'] == False) 
     
    ]

In [ ]:
import requests
import xml.etree.ElementTree as ET

def isStillViolation(row):
    
    if row['object'].startswith("_:") or row['object'].startswith("http://"):
        return None
    
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"

    # SPARQL query
    query = f"""
            Select (count(distinct ?s) as ?total)
            {{
              ?s <{row['wdt_prop']}> ?o.
              FILTER (?o = "{row['object']}")
            }} LIMIT 2
            """

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)

        # Define the namespace used in the XML
        namespace = {'ns': 'http://www.w3.org/2005/sparql-results#'}

        # Find the 'literal' element containing the 'total' value
        literal_element = root.find('.//ns:literal', namespace)

        # Extract the text from the 'literal' element and convert to an integer
        total_value = int(literal_element.text) if literal_element is not None else 0

        # Return True if total is 1, False otherwise
        result = total_value > 1
        
        return result
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None

# Example usage
print(isStillViolation(df_distinct_values.iloc[39]))

In [ ]:
#df_distinct_values["still_violation"] = None

In [ ]:
df_distinct_values.dtypes

In [ ]:
# Filter the DataFrame based on multiple conditions
filtered_df = df_distinct_values[(df_distinct_values['Constraint Deleted'] == False) &
                 (df_distinct_values['Constraint Deprecated'] == False) &
                 (df_distinct_values['Included as Exception'] == False) &
                 (df_distinct_values['A-box wdt statement Deleted'] == False) &
                 (df_distinct_values['Indirect Correction by Exception'] == False) &
                 (df_distinct_values['Indirect Correction by Statement Deletion'] == False)]

In [ ]:
filtered_df

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm
import os

checkpoint_file = "checkpoint_distinct_values.csv"
# Process the rows with progress bar
for index, row in tqdm(filtered_df.iterrows(), total=len(filtered_df)):
    
    # Check for unprocessed rows
    if pd.isna(row['still_violation']):
        result = isStillViolation(row)
        filtered_df.at[index, 'still_violation'] = result

    # Save a checkpoint every 10,000 rows
    if index % 5000 == 0:
        filtered_df.to_csv(checkpoint_file, index=True)
        print(f"Checkpoint saved at row {index}.")

# Save the final output
filtered_df.to_csv(checkpoint_file, index=True)
print("Processing complete. Final output saved.")

In [ ]:
# 36043
filtered_df.to_csv(checkpoint_file, index=True)

In [ ]:
filtered_df.iloc[15165]

In [ ]:
import pandas as pd

# Read the first CSV file into a DataFrame
df_distinct_values = pd.read_csv("distinct_values_correcoes.csv")

df_distinct_values